In [ ]:
import numpy as np
import pandas as pd

In [ ]:

#read histone signal and rename colname
df = pd.read_table("/path/to/cutandtag/data/total_degs_promoter_histonesignal.txt",header=None)
df.columns = ["chr", "start", "end", "wt-k27ac_1", "wt-k27ac_2",'wt-4me3_1','wt-4me3_2','wt-27me3_1','wt-27me3_2', "wt-input_1", "wt-input_2", "pi-k27ac_1", "pi-k27ac_2",'pi-4me3_1','pi-4me3_2','pi-27me3_1','pi-27me3_2',"pi-input_1", "pi-input_2","DEGs"]
df.head()

In [ ]:
#calibrate depth
df["wt-k27ac_1nor"] = df["wt-k27ac_1"]/47
df["wt-k27ac_2nor"] = df["wt-k27ac_2"]/53
df["wt-4me3_1nor"] = df["wt-4me3_1"]/37
df["wt-4me3_2nor"] = df["wt-4me3_2"]/34
df["wt-k27me3_1nor"] = df["wt-27me3_1"]/63
df["wt-k27me3_2nor"] = df["wt-27me3_2"]/73
df["wt-igg_1nor"] = df["wt-input_1"]/10
df["wt-igg_2nor"] = df["wt-input_2"]/10
df["pi-k27ac_1nor"] = df["pi-k27ac_1"]/38
df["pi-k27ac_2nor"] = df["pi-k27ac_2"]/59
df["pi-4me3_1nor"] = df["pi-4me3_1"]/13
df["pi-4me3_2nor"] = df["pi-4me3_2"]/58
df["pi-k27me3_1nor"] = df["pi-27me3_1"]/26
df["pi-k27me3_2nor"] = df["pi-27me3_2"]/46
df["pi-igg_1nor"] = df["pi-input_1"]/8
df["pi-igg_2nor"] = df["pi-input_2"]/9
df.head()

In [ ]:
#compute Mock replicate average
df['wt-k27ac_mean'] = df[['wt-k27ac_1nor', 'wt-k27ac_2nor']].mean(axis=1)
df['wt-4me3_mean'] = df[['wt-4me3_1nor', 'wt-4me3_2nor']].mean(axis=1)
df['wt-k27me3_mean'] = df[['wt-k27me3_1nor', 'wt-k27me3_2nor']].mean(axis=1)
df['wt-igg_mean'] = df[['wt-igg_1nor', 'wt-igg_2nor']].mean(axis=1)

In [ ]:
#compute TGEV replicate average
df['pi-k27ac_mean'] = df[['pi-k27ac_1nor', 'pi-k27ac_2nor']].mean(axis=1)
df['pi-4me3_mean'] = df[['pi-4me3_1nor', 'pi-4me3_2nor']].mean(axis=1)
df['pi-k27me3_mean'] = df[['pi-k27me3_1nor', 'pi-k27me3_2nor']].mean(axis=1)
df['pi-igg_mean'] = df[['pi-igg_1nor', 'pi-igg_2nor']].mean(axis=1)

In [ ]:
#remove background effect
df["wt-k27ac_change"] = np.log2((df["wt-k27ac_mean"] + 1e-3)/(df["wt-igg_mean"] + 1e-3))
df["wt-4me3_change"] = np.log2((df["wt-4me3_mean"] + 1e-3)/(df["wt-igg_mean"] + 1e-3))
df["wt-k27me3_change"] = np.log2((df["wt-k27me3_mean"] + 1e-3)/(df["wt-igg_mean"] + 1e-3))
df["pi-k27ac_change"] = np.log2((df["pi-k27ac_mean"] + 1e-3)/(df["pi-igg_mean"] + 1e-3))
df["pi-4me3_change"] = np.log2((df["pi-4me3_mean"] + 1e-3)/(df["pi-igg_mean"] + 1e-3))
df["pi-k27me3_change"] = np.log2((df["pi-k27me3_mean"] + 1e-3)/(df["pi-igg_mean"] + 1e-3))
df.head()

In [ ]:
#remove noise
df.loc[df['wt-k27ac_change'] < 0, 'wt-k27ac_change'] = 0
df.loc[df['wt-4me3_change'] < 0, 'wt-4me3_change'] = 0
df.loc[df['wt-k27me3_change'] < 0, 'wt-k27me3_change'] = 0
df.loc[df['pi-k27ac_change'] < 0, 'pi-k27ac_change'] = 0
df.loc[df['pi-4me3_change'] < 0, 'pi-4me3_change'] = 0
df.loc[df['pi-k27me3_change'] < 0, 'pi-k27me3_change'] = 0
df.head()

In [ ]:
#save h3k27ac calibrated result
h3k27ac = df[["chr","start","end","wt-k27ac_change","pi-k27ac_change","DEGs"]] 
h3k27ac.head()
h3k27ac.to_csv("/path/to/cutandtag/data/h3k27ac_change.txt", sep="\t", index=False)

In [ ]:
#save h3k27me3 calibrated result
h3k27me3 = df[["chr","start","end","wt-k27me3_change","pi-k27me3_change","DEGs"]] 
h3k27me3.head()
h3k27me3.to_csv("/path/to/cutandtag/data/h3k27me3_change.txt", sep="\t", index=False)

In [ ]:
#save h3k4me3 calibrated result
h3k4me3 = df[["chr","start","end","wt-4me3_change","pi-4me3_change","DEGs"]] 
h3k4me3.head()
h3k4me3.to_csv("/path/to/cutandtag/data/h3k4me3_change.txt", sep="\t", index=False)

In [ ]:
#compute TGEV/Mock log2FC
h3k27ac = h3k27ac[~((h3k27ac['wt-k27ac_change'] <= 0) & (h3k27ac['pi-k27ac_change'] <= 0))]
h3k27ac['change'] = h3k27ac['pi-k27ac_change'] - h3k27ac['wt-k27ac_change']
h3k27ac.head()

In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator
from scipy.stats import mannwhitneyu, normaltest

In [ ]:
#plot
fig, ax = plt.subplots(figsize=(3,4))
ax = sns.boxplot(x='DEGs', y='change', hue="DEGs", data=h3k27ac, showfliers=False,width=0.3,palette=("#304172","#bfbfbf", "#c0271d"))
pairs = [('down','none'),('none','up')]
annotator = Annotator(ax, pairs, x='DEGs', y='change', hue="DEGs", data=h3k27ac, showfliers=False,width=0.3,palette=("#304172","#bfbfbf", "#c0271d"))
annotator.configure(test='Mann-Whitney', text_format='star',line_height=0.03, line_width=1)
annotator.apply_and_annotate()
plt.title("H3K27ac FC of DEGs")
plt.savefig("/path/to/cutandtag/data/degs_27ac_fc.pdf")
plt.tight_layout()
plt.show()

In [ ]:
h3k27me3 = h3k27me3[~((h3k27me3['wt-k27me3_change'] <= 0) & (h3k27me3['pi-k27me3_change'] <= 0))]
h3k27me3['change'] = h3k27me3['pi-k27me3_change'] - h3k27me3['wt-k27me3_change']
h3k27me3.head()

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
ax = sns.boxplot(x='DEGs', y='change', hue="DEGs", data=h3k27me3, showfliers=False,width=0.3,palette=("#304172","#bfbfbf", "#c0271d"))
pairs = [('up','none'),('none','down')]
annotator = Annotator(ax, pairs, x='DEGs', y='change', hue="DEGs", data=h3k27me3, showfliers=False,width=0.3,palette=("#304172","#bfbfbf", "#c0271d"))
annotator.configure(test='Mann-Whitney', text_format='star',line_height=0.03, line_width=1)
annotator.apply_and_annotate()
plt.title("H3K27me3 FC of DEGs")
plt.savefig("/path/to/cutandtag/data/degs_27me3_fc.pdf")
plt.tight_layout()
plt.show()

In [ ]:
h3k4me3 = h3k4me3[~((h3k4me3['wt-4me3_change'] <= 0) & (h3k4me3['pi-4me3_change'] <= 0))]
h3k4me3['change'] = h3k4me3['pi-4me3_change'] - h3k4me3['wt-4me3_change']
h3k4me3.head()

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
ax = sns.boxplot(x='DEGs', y='change', hue="DEGs", data=h3k4me3, showfliers=False,width=0.3,palette=("#304172","#bfbfbf", "#c0271d"))
pairs = [('down','none'),('none','up')]
annotator = Annotator(ax, pairs, x='DEGs', y='change', hue="DEGs", data=h3k4me3, showfliers=False,width=0.3,palette=("#304172","#bfbfbf", "#c0271d"))
annotator.configure(test='Mann-Whitney', text_format='star',line_height=0.03, line_width=1)
annotator.apply_and_annotate()
plt.title("H3K4me3 FC of DEGs")
plt.savefig("/path/to/cutandtag/data/degs_4me3_fc.pdf")
plt.tight_layout()
plt.show()